In [ ]:
import json
import threading

from flask import Flask
import paho.mqtt.client as mqtt

# MQTT settings
MQTT_BROKER = "localhost"
MQTT_PORT = 1883
MQTT_TOPIC = "hazard/data"

# Flask app
app = Flask(__name__)

# latest sensor values
latest_data = {
    "timestamp": "-",
    "gas": 0,
    "temp": 0,
    "temp_F": 0,
    "hum": 0,
    "flame": False,
    "score": 0,
    "status": "SAFE"
}

# when MQTT connects
def on_connect(client, userdata, flags, rc):
    print("Connected to MQTT")
    client.subscribe(MQTT_TOPIC)

# when MQTT receives data
def on_message(client, userdata, msg):
    global latest_data
    try:
        latest_data = json.loads(msg.payload.decode())
        print("Received:", latest_data)
    except Exception as e:
        print("Error:", e)

# run MQTT in background
def mqtt_loop():
    client = mqtt.Client()
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(MQTT_BROKER, MQTT_PORT, 60)
    client.loop_forever()

# webpage
@app.route("/")
def home():
    return f"""
    <html>
    <head>
        <title>Hazard Dashboard</title>
        <meta http-equiv="refresh" content="2">
    </head>
    <body>
        <h1>Smart Hazard Detection Dashboard</h1>
        <p><b>Time:</b> {latest_data['timestamp']}</p>
        <p><b>Gas:</b> {latest_data['gas']}</p>
        <p><b>Temperature:</b> {latest_data['temp']} °C / {latest_data['temp_F']} °F</p>
        <p><b>Humidity:</b> {latest_data['hum']} %</p>
        <p><b>Flame detected:</b> {latest_data['flame']}</p>
        <p><b>Hazard score:</b> {latest_data['score']}</p>
        <p><b>Status:</b> {latest_data['status']}</p>
    </body>
    </html>
    """

# start everything
if __name__ == "__main__":
    thread = threading.Thread(target=mqtt_loop)
    thread.daemon = True
    thread.start()

    print("Starting dashboard...")
    app.run(host="0.0.0.0", port=5050)